Medical professionals often summarize patient encounters in transcripts written in natural language, which include details about symptoms, diagnosis, and treatments. These transcripts can be used for other medical documentation, such as for insurance purposes, but as they are densely packed with medical information, extracting the key data accurately can be challenging.  

You and your team at Lakeside Healthcare Network have decided to leverage the OpenAI API to automatically extract medical information from these transcripts and automate the matching with the appropriate ICD-10 codes. ICD-10 codes are a standardized system used worldwide for diagnosing and billing purposes, such as insurance claims processing.

## The Data
The dataset contains anonymized medical transcriptions categorized by specialty.

## transcriptions.csv
| Column     | Description              |
|------------|--------------------------|
| `"medical_specialty"` | The medical specialty associated with each transcription.  |
| `"transcription"` | Detailed medical transcription texts, with insights into the medical case. |

In [3]:
# Import the necessary libraries
import pandas as pd
from openai import OpenAI
import json

In [4]:
# Load the data
df = pd.read_csv("data/transcriptions.csv")
df.head()

,medical_specialty,transcription
0,Allergy / Immunology,"SUBJECTIVE:, This 23-year-old white female pr..."
1,Orthopedic,"CHIEF COMPLAINT:, Achilles ruptured tendon.,H..."
2,Bariatrics,"PREOPERATIVE DIAGNOSIS: , Morbid obesity.,POST..."
3,Cardiovascular / Pulmonary,"PREOPERATIVE DIAGNOSES,Airway obstruction seco..."
4,Urology,"CHIEF COMPLAINT:, Urinary retention.,HISTORY ..."


In [ ]:
# Initialize the OpenAI client
client = OpenAI()

In [ ]:
function_definition = []

# Extract Patient Data function
function_extract_patient_data = {'type': 'function',
  'function': {
    'name': 'extract_patient_data',
    'description': 'Extract the age and recommended treatment or procedure information from transcription, and get medical specialty info from medical_specialty. And find ICD Code for correspoding data',
    'parameters': {
        'type': 'object',
        'properties': {
            'patients': {
                'type': 'array',
                'description': 'List of extracted patient data',
                'items': {
                    'type': 'object',
                    'properties': {
                        'age': {'type': 'string', 'description': 'Age'}, 
                        'treatment': {'type': 'string', 'description': 'Recommended treatment or procedure information'},
                        'medical_specialty': {'type': 'string', 'description': 'Medical Speciality'},
                        'ICD_code': {'type': 'string', 'description': 'ICD code'}
                    },
                'required': ['age', 'treatment', 'medical_specialty', 'ICD_code']
            }
        }
    },
        'required': ['patients']
  }
}
}

function_definition.append(function_extract_patient_data)

results = []
for i, row in df.iterrows():
    response= client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", f"content": "I have this dataframe {row} with two columns: medical_specialty, transcription. The medical_specialty column has the Medical Speciality info which same for every patient in the transcription column. The transcription column have information about different patients, so I want to extract all the age, the recommended treatment or procedure data from the column. And you should add the ICD-10 code for the corresponding data using the Treatment and Medical Speciality information. Result should be json with 4 keys which are in order: Age, Treatment, ICD Code, Medical Speciality"}],
        tools=function_definition,
        tool_choice={'type': 'function', 'function': {'name': 'extract_patient_data'}},
        response_format={"type": "json_object"},
        temperature=0
    )


    patients = json.loads(response.choices[0].message.tool_calls[0].function.arguments)["patients"]

    for patient_data in patients:
        results.append(patient_data)


In [ ]:
# Resulting df
df_structured = pd.DataFrame(results)
df_structured

,age,treatment,medical_specialty,ICD_code
0,25,Physical therapy for knee pain,Orthopedics,M17.9
1,30,Medication for hypertension,Cardiology,I10
2,45,Surgery for gallbladder removal,General Surgery,K80.20
3,60,Insulin therapy for diabetes,Endocrinology,E11.9
4,25,Physical Therapy,Orthopedics,M54.5
5,30,Medication,Cardiology,I10
6,45,Surgery,General Surgery,K35
7,60,Chemotherapy,Oncology,C50.9
8,25,Physical therapy,Orthopedics,M54.5
9,30,Surgery,General Surgery,K35
